In [1]:
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from dotenv import load_dotenv
import os
from crewai import Agent, Task, Crew, Process, LLM
import os
from langchain_openai import ChatOpenAI
from crewai_tools import MCPServerAdapter
from dotenv import load_dotenv
# from langchain_groq import ChatGroq

import os
from crewai import LLM
from langchain_openai import ChatOpenAI

In [2]:
load_dotenv()

True

In [3]:
from crew_result_interpreter import ResultInterpreterCrew

In [4]:
model_ids = [
    "groq/compound",
    "groq/compound-mini",
    "llama-3.1-8b-instant",
    "llama-3.3-70b-versatile",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "meta-llama/llama-prompt-guard-2-22m",
    "meta-llama/llama-prompt-guard-2-86m",
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
    "openai/gpt-oss-safeguard-20b",
    "qwen/qwen3-32b",
]

In [22]:
model_id = model_ids[3]
model_id

'llama-3.3-70b-versatile'

In [23]:
llm = ChatOpenAI(
    openai_api_base="https://api.groq.com/openai/v1",
    openai_api_key=os.environ.get("GROQ_API_KEY"),
    temperature=0,
    model_name=f"groq/{model_id}",
    top_p=1,
    max_retries=3,
    request_timeout=60,
)

In [24]:
crew = ResultInterpreterCrew(llm=llm, max_crew_rpm=1)

In [25]:
crew.default_llm

ChatOpenAI(output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x7f1e777a0470>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7f1e3580ac30>, root_client=<openai.OpenAI object at 0x7f1e77fc4650>, root_async_client=<openai.AsyncOpenAI object at 0x7f1e3578a8d0>, model_name='groq/llama-3.3-70b-versatile', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.groq.com/openai/v1', openai_proxy=None, request_timeout=60.0, max_retries=3, top_p=1.0, stream_chunk_timeout=120.0)

In [26]:
len(crew.mcp_adapter.tools)

8

In [27]:
from utils_db_helper import (
    get_db_schema,
    run_query,
    clean_sql_query,
    set_database_path,
    get_database_path
)

In [28]:
reviewed_sql = 'SELECT score FROM table_name_89 WHERE visitor = "toronto" AND record = "29-17-8"'

In [29]:
db_path = get_database_path()

In [30]:
db_path

'/home/ridwanfatur/portfolio/sql-assistant-mcp/test_cases/001/database.sqlite'

In [31]:
query_result = run_query(reviewed_sql, db_path)

In [32]:
query_result

'score\n  5-2'

In [33]:
user_input = 'Name the score for toronto visitor and record of 29-17-8'

In [34]:
interpretation_task = Task(
    description=f"""
    Interpret these SQL query results for business users:
    
    Original Request: {user_input}
    SQL Query: {reviewed_sql}
    Query Results: {query_result}
    
    Provide a business-friendly interpretation with insights and recommendations.
    """,
    agent=crew.result_interpreter_agent(),
    expected_output="A comprehensive business interpretation of the query results",
)

In [35]:
interpretation_crew = Crew(
    agents=[crew.result_interpreter_agent()],
    tasks=[interpretation_task],
)  

In [36]:
interpretation_result = interpretation_crew.kickoff()

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Business Intelligence Analyst                                                                           │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Interpret these SQL query results for business users:                                                      │
│                                                                                                                 │
│      Original Request: Name the score for toronto visitor and record of 29-17-8                                 │
│      SQL Query: SELECT score FROM table_name_89 WHERE visitor = "toronto" AND record = "29-17-8"                │
│      Query Results: score                                                                                       │
│    5-2                                                                                                          │
│                                                                                                                 │
│      Provide a business-friendly interpretation with insights and recommendations.                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Business Intelligence Analyst                                                                           │
│                                                                                                                 │
│  Thought: Thought: The SQL query results show the score for the Toronto visitor with a record of 29-17-8,       │
│  which is 5-2. To provide a business-friendly interpretation, I need to understand the context of the data and  │
│  the implications of the results.                                                                               │
│                                                                                                                 │
│  Using Tool: run_sql_query                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "query": "SELECT score FROM table_name_89 WHERE visitor = \"toronto\" AND record = \"29-17-8\""              │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  score                                                                                                          │
│    5-2                                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Business Intelligence Analyst                                                                           │
│                                                                                                                 │
│  Thought: Thought: The query results confirm that the score for the Toronto visitor with a record of 29-17-8    │
│  is indeed 5-2. To put this into perspective, I should consider the overall performance of the team and the     │
│  visitor's record.                                                                                              │
│                                                                                                                 │
│  Using Tool: count_rows                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "table_name": "table_name_89"                                                                                │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Table 'table_name_89' has 16 rows                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Business Intelligence Analyst                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The SQL query results show that the score for the Toronto visitor with a record of 29-17-8 is 5-2. This        │
│  suggests that the team had a strong performance, with 5 goals scored and 2 goals conceded. The record of       │
│  29-17-8 indicates a relatively successful season, with 29 wins, 17 losses, and 8 draws.                        │
│                                                                                                                 │
│  From a business perspective, this information can be used to analyze the team's strengths and weaknesses, as   │
│  well as the impact of the visitor's record on the game's outcome. The score of 5-2 may indicate that the team  │
│  has a strong offense, but may also have some vulnerabilities in their defense.                                 │
│                                                                                                                 │
│  Key metrics:                                                                                                   │
│  - Score: 5-2                                                                                                   │
│  - Record: 29-17-8                                                                                              │
│  - Total rows in the table: 16                                                                                  │
│                                                                                                                 │
│  Trends:                                                                                                        │
│  - The team's strong performance, with 29 wins and a score of 5-2, suggests a competitive edge.                 │
│  - The visitor's record of 29-17-8 indicates a relatively successful season, but also some areas for            │
│  improvement.                                                                                                   │
│                                                                                                                 │
│  Actionable recommendations:                                                                                    │
│  - Analyze the team's offense and defense strategies to identify areas for improvement.                         │
│  - Consider the impact of the visitor's record on the game's outcome and adjust strategies accordingly.         │
│  - Use data from the 16 rows in the table to gain a deeper understanding of the team's performance and make     │
│  informed decisions.                                                                                            │
│                                                                                                                 │
│  Overall, the query results provide valuable insights into the team's performance and the visitor's record,     │
│  which can be used to inform business decisions and drive success.                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [37]:
business_interpretation = str(interpretation_result).strip()

In [38]:
business_interpretation

"The SQL query results show that the score for the Toronto visitor with a record of 29-17-8 is 5-2. This suggests that the team had a strong performance, with 5 goals scored and 2 goals conceded. The record of 29-17-8 indicates a relatively successful season, with 29 wins, 17 losses, and 8 draws. \n\nFrom a business perspective, this information can be used to analyze the team's strengths and weaknesses, as well as the impact of the visitor's record on the game's outcome. The score of 5-2 may indicate that the team has a strong offense, but may also have some vulnerabilities in their defense. \n\nKey metrics:\n- Score: 5-2\n- Record: 29-17-8\n- Total rows in the table: 16\n\nTrends:\n- The team's strong performance, with 29 wins and a score of 5-2, suggests a competitive edge.\n- The visitor's record of 29-17-8 indicates a relatively successful season, but also some areas for improvement.\n\nActionable recommendations:\n- Analyze the team's offense and defense strategies to identify ar

In [42]:
interpretation_result.token_usage.__dict__

{'total_tokens': 3755,
 'prompt_tokens': 2335,
 'cached_prompt_tokens': 0,
 'completion_tokens': 1420,
 'successful_requests': 3}